# Wanamaker Quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mbagalman/wanamaker/blob/master/notebooks/quickstart.ipynb)

This notebook runs Wanamaker end-to-end on the bundled `public_benchmark`
dataset — diagnose, fit, executive summary, Trust Card, and the HTML showcase
report — without leaving the browser.

**Time:** about 10 minutes including the ~3-minute model fit.

**Requirements:** Python 3.11+. Colab's standard runtime works.

Wanamaker makes no network calls during analysis. Installation is the only
step that hits the network.

## 1 — Install Wanamaker

In [ ]:
%pip install -q git+https://github.com/mbagalman/wanamaker.git

On Colab you may need to restart the runtime after installation so PyMC's
compiled extensions are picked up (`Runtime → Restart runtime`). The next
cell verifies the install.

In [ ]:
import wanamaker

print(f"wanamaker {wanamaker.__version__}")

## 2 — Look at the example data

Wanamaker ships with `public_benchmark`, a synthetic weekly dataset with eight
media channels and four controls. It is safe to inspect publicly.

In [ ]:
import importlib.resources as resources
import pandas as pd

example_root = resources.files("wanamaker._examples.public_benchmark")
csv_path = example_root.joinpath("public_example.csv")
df = pd.read_csv(csv_path)
print(f"{len(df)} rows × {len(df.columns)} columns")
df.head()

## 3 — Run the end-to-end demo

This chains the readiness diagnostic, a quick-mode fit, the Markdown report,
and the HTML showcase. On Colab's CPU runtime expect 3–5 minutes.

In [ ]:
!wanamaker run --example public_benchmark

## 4 — View the showcase

The `run` command auto-renders an HTML showcase — a single self-contained file
you can email or print. It includes the executive summary, channel contributions
with 95% credible intervals, ROI, and the Trust Card.

In [ ]:
from pathlib import Path

from IPython.display import HTML, display

_runs_dir = Path(".wanamaker/runs")
_latest_run = max(_runs_dir.iterdir(), key=lambda p: p.stat().st_mtime)
_showcase_path = _latest_run / "showcase.html"
print(f"Showcase: {_showcase_path}")
display(HTML(_showcase_path.read_text(encoding="utf-8")))

## 5 — Read the Markdown report

The same content also lives at `.wanamaker/runs/<run_id>/report.md` — better
suited for git, code review, or pasting into Slack.

In [ ]:
from IPython.display import Markdown

_report_path = _latest_run / "report.md"
display(Markdown(_report_path.read_text(encoding="utf-8")))

## 6 — Bring your own data

To run Wanamaker against your own dataset:

1. Upload a weekly CSV via Colab's **Files → Upload to session storage**.
2. Write a YAML config naming your date column, target column, media spend
   columns, and any controls. See
   [`benchmark_data/public_example.yaml`](https://github.com/mbagalman/wanamaker/blob/master/benchmark_data/public_example.yaml)
   for the schema.
3. Run:

   ```python
   !wanamaker diagnose your_data.csv --config your_config.yaml
   !wanamaker fit --config your_config.yaml
   !wanamaker showcase --run-id <run_id_printed_by_fit>
   ```

Treat sensitive spend data carefully on Colab. Wanamaker itself makes no
outbound network calls during analysis, but Colab is hosted infrastructure.
Run on your own machine when in doubt — the same commands work locally.

## What to read next

- [Quickstart](https://github.com/mbagalman/wanamaker/blob/master/docs/quickstart.md) — the same flow on the command line
- [Analyst's Guide](https://github.com/mbagalman/wanamaker/blob/master/docs/analyst_guide.md)
- [Risk-Adjusted Allocation Ramps](https://github.com/mbagalman/wanamaker/blob/master/docs/risk_adjusted_allocation.md) — design of the `recommend-ramp` command
- [Comparison to Robyn / Meridian / PyMC-Marketing](https://github.com/mbagalman/wanamaker/blob/master/docs/comparison.md)